# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [1]:

print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 153.29 GB
MemAvailable: 966.58 GB
Free GPU Memory (GB): 34.7754

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################

Free GPU Memory (GB): 34.7754. Context: Warm up notebook.


## 2. Loading Datasets

### 2.1 T-Rex

In [3]:
import os

print("\n################################")
print("Setting up T-REX...")
print("################################\n")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.

from datasets import load_dataset
ds = load_dataset("relbert/t_rex")
ds



################################
Setting up T-REX...
################################



DatasetDict({
    train: Dataset({
        features: ['relation', 'head', 'tail', 'title', 'text'],
        num_rows: 1274264
    })
    validation: Dataset({
        features: ['relation', 'head', 'tail', 'title', 'text'],
        num_rows: 318566
    })
    test: Dataset({
        features: ['relation', 'head', 'tail', 'title', 'text'],
        num_rows: 122
    })
})

## 3. FKTC Evaluation

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.model.eval()

    def generate_response(self, query, max_new_tokens):
        input_ids = self.tokenizer.encode(query, return_tensors='pt').to("cuda")
        generation_config = {
            "temperature": 0.1,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 1,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }

        with torch.no_grad():
            outputs = self.model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        probabilities = self.extract_probabilities(outputs)
        return output_text, probabilities

    def extract_probabilities(self, outputs):
        probabilities = []
        for score in outputs.scores:
            probs = torch.softmax(score[0], dim=-1)
            top_prob, top_idx = torch.max(probs, dim=-1)
            probabilities.append((self.tokenizer.decode(top_idx), top_prob.item()))
        return probabilities

# Example usage:
model_names = [
    "bigscience/bloomz-560m",
    "bigscience/bloomz-1b1",
    "TinyLlama/TinyLlama_v1.1",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    # "facebook/galactica-1.3b",
    "meta-llama/Meta-Llama-3-8B",
]
queries = [
    "What is the location of Simcoe Composite School?",
    "musical. What is Alan Turing's area of expertise?",
    "Latin. What is the native language of Louis Jules Trochu?"
]
true_answers = [
    "Canada.",
    "logic",
    "French"
]
max_new_tokens_list = [10, 25]

# Initialize a list to store the results
results = []

for query, true_answer in zip(queries, true_answers):
    for model_name in model_names:
        generator = ResponseGenerator(model_name)
        for max_new_tokens in max_new_tokens_list:
            output_text, probabilities = generator.generate_response(query, max_new_tokens)
            if output_text.startswith(query):
                output_text = output_text[len(query):].strip()
            results.append({
                "Model": model_name,
                "Query": query,
                "Max New Tokens": max_new_tokens,
                "True Answer": true_answer,
                "Output": output_text,
                "Probabilities": probabilities
            })

# Convert the results list to a pandas DataFrame
df = pd.DataFrame(results)

# Print the DataFrame
print(df)

# Export the DataFrame to an Excel file
df.to_excel("results.xlsx", index=False)


Initializing MonitorEvaluator...
Loading model TinyLlama/TinyLlama_v1.1...
Loading tokenizer TinyLlama/TinyLlama_v1.1...
Loading data from /nfs/students/daro/data/MONITOR/FKTC...
Loading files ['P101-subclass.json', 'P103-subclass.json', 'P108-subclass.json', 'P127-subclass.json', 'P1376-subclass.json', 'P1412-subclass.json', 'P159-subclass.json', 'P17-subclass.json', 'P176-subclass.json', 'P178-subclass.json', 'P19-subclass.json', 'P20-subclass.json', 'P264-subclass.json', 'P27-subclass.json', 'P276-subclass..json', 'P30-subclass.json', 'P364-subclass.json', 'P37-subclass.json', 'P495-subclass.json', 'P740-subclass.json']...
Max new tokens: 10


Evaluating FKTC data...
Loading data from P101-subclass.json...
Processing file P101-subclass.json, entry 0...
Processing file P101-subclass.json, entry 1...
Loading data from P103-subclass.json...
Processing file P103-subclass.json, entry 0...
Processing file P103-subclass.json, entry 1...
Prompt: What is Alan Turing's area of expertise?
Answer: 200000000
True Object: logic
Prompt: musical. What is Alan Turing's area of expertise?
Answer: Alan Turing's area of expertise was
True Object: logic
Prompt: What is John Vincent Atanasoff's area of expertise?
Answer: AtApr 11
True Object: mathematics
Prompt: abortion. What is John Vincent Atanasoff's area of expertise?
Answer: John Vincent Atanasoff's area of
True Object: mathematics
Prompt: What is the native language of Louis Jules Trochu?
Answer: Today we are going to talk about
True Object: French
Prompt: Latin. What is the native language of Louis Jules Trochu?
Answer: What is the native language of Louis Jules Troch
True Object: French


Evaluating FKTC data...
Loading data from P101-subclass.json...
Processing file P101-subclass.json, entry 0...
Processing file P101-subclass.json, entry 1...
Loading data from P103-subclass.json...
Processing file P103-subclass.json, entry 0...
Processing file P103-subclass.json, entry 1...
Prompt: What is Alan Turing's area of expertise?
Answer: 2000000000000000000
True Object: logic
Prompt: musical. What is Alan Turing's area of expertise?
Answer: What is Alan Turing's area of expertise What is Alan Turing's area
True Object: logic
Prompt: What is John Vincent Atanasoff's area of expertise?
Answer: AtApr 11 2019 · The 201
True Object: mathematics
Prompt: abortion. What is John Vincent Atanasoff's area of expertise?
Answer: What is John Vincent Atanasoff's area of expertise What is John Vincent Atanas
True Object: mathematics
Prompt: What is the native language of Louis Jules Trochu?
Answer: Today we are going to talk about how to make money online in 20
True Object: French
Prompt: La

Evaluating FKTC data...
Loading data from P101-subclass.json...
Processing file P101-subclass.json, entry 0...
Processing file P101-subclass.json, entry 1...
Loading data from P103-subclass.json...
Processing file P103-subclass.json, entry 0...
Processing file P103-subclass.json, entry 1...
Prompt: What is Alan Turing's area of expertise?
Answer: 20000000000000000000000000000
True Object: logic
Prompt: musical. What is Alan Turing's area of expertise?
Answer: What is Alan Turing's area of expertise What is Alan Turing's area of expertise What is Alan Turing'
True Object: logic
Prompt: What is John Vincent Atanasoff's area of expertise?
Answer: AtApr 11 2019 · The 2019-2020 school year will be
True Object: mathematics
Prompt: abortion. What is John Vincent Atanasoff's area of expertise?
Answer: What is John Vincent Atanasoff's area of expertise What is John Vincent Atanasoff's area of expertise What is
True Object: mathematics
Prompt: What is the native language of Louis Jules Trochu?
A

Evaluating FKTC data...
Loading data from P101-subclass.json...
Processing file P101-subclass.json, entry 0...
Processing file P101-subclass.json, entry 1...
Loading data from P103-subclass.json...
Processing file P103-subclass.json, entry 0...


KeyboardInterrupt: 

In [ ]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3-8B"
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
pipeline = transformers.pipeline(
  "text-generation", model=model_id, model_kwargs={"torch_dtype": torch.bfloat16}, device_map="cuda"
)
# output = pipeline("What is the capital city of Hungary?", max_new_tokens=15)
output = pipeline("Which city is Chandos Records's corporate headquarters located?", max_new_tokens=100)
answer = output[0]['generated_text']
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(output)

In [7]:
results = evaluator.evaluate()
for results_list in results:
    for result in results_list:
        if result['is_correct']:
            print(f"Prompt: {result['prompt']}")
            print(f"Answer: {result['answer']}")
            print(f"True Object: {result['true_object']}")

Evaluating FKTC data...
Processing file P101-subclass.json...
Loading data from P101-subclass.json...
Processing entry 0...
Entry: {'subject': 'Alan Turing', 'object': 'logic', 'taxonomy': ['musical', 'car', 'ecology', 'photography', 'Balkans']}
Processing entry 1...
Entry: {'subject': 'John Vincent Atanasoff', 'object': 'mathematics', 'taxonomy': ['abortion', 'theology', 'hunting', 'archaeology', 'geology']}
Processing file P103-subclass.json...
Loading data from P103-subclass.json...
Processing entry 0...
Entry: {'subject': 'Louis Jules Trochu', 'object': 'French', 'taxonomy': ['Latin', 'Malayalam', 'Croatian', 'Welsh', 'Bengali']}
Processing entry 1...
Entry: {'subject': 'Valeria Bruni Tedeschi', 'object': 'French', 'taxonomy': ['Croatian', 'Serbian', 'Chinese', 'Welsh', 'Cantonese']}
Prompt: Latin. What is the native language of Louis Jules Trochu?
Answer: Louis Jules Trochu (French pronunciation: [lwi ʒ
True Object: French


In [17]:
print(sum([sum([result['is_correct'] for result in results_list]) for results_list in results]))
print(sum([sum([not result['is_correct'] for result in results_list]) for results_list in results]))

66
30


In [6]:
"John Vincent Atanasoff works in the computer industry <|user|>".lower()

'john vincent atanasoff works in the computer industry <|user|>'

In [15]:
for results_list in results:
  for result in results_list:
    if not result['is_correct']:
      print(f"Prompt: {result['prompt']}")
      print(f"Answer: {result['answer']}")
      print(f"True Object: {result['true_object']}")

Prompt: Which industry does Alan Turing work in?
Answer: Alan Turing worked in the military intelligence branch of the British Army during World War II
True Object: logic
Prompt: [musical]. Which industry does Alan Turing work in?
Answer: Alan Turing worked in the military intelligence branch of the British Army during World War II
True Object: logic
Prompt: [car]. Which industry does Alan Turing work in?
Answer: Alan Turing works in the computer science industry <|user|>
True Object: logic
Prompt: What is John Vincent Atanasoff's area of expertise?
Answer: John Vincent Atanasoff's area of expertise is electrical engineering <
True Object: mathematics
Prompt: [abortion]. What is John Vincent Atanasoff's area of expertise?
Answer: John Vincent Atanasoff's area of expertise is computer science <|
True Object: mathematics
Prompt: [theology]. What is John Vincent Atanasoff's area of expertise?
Answer: John Vincent Atanasoff's area of expertise is computer science <|
True Object: mathematic

In [1]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3-8B"
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
pipeline = transformers.pipeline(
  "text-generation", model=model_id, model_kwargs={"torch_dtype": torch.bfloat16}, device_map="cuda"
)
# output = pipeline("What is the capital city of Hungary?", max_new_tokens=15)
output = pipeline("Which city is Chandos Records's corporate headquarters located?", max_new_tokens=100)
answer = output[0]['generated_text']
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(output)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Free GPU Memory (GB): 39.3896


Loading checkpoint shards: 100%|██████████| 4/4 [01:15<00:00, 18.93s/it]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Free GPU Memory (GB): 23.7812
[{'generated_text': "Which city is Chandos Records's corporate headquarters located?"}]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [3]:
!ls /nfs/students/daro/data/MONITOR/FKTC/

P101-subclass.json   P17-subclass.json	 P276-subclass..json
P103-subclass.json   P176-subclass.json  P30-subclass.json
P108-subclass.json   P178-subclass.json  P364-subclass.json
P127-subclass.json   P19-subclass.json	 P37-subclass.json
P1376-subclass.json  P20-subclass.json	 P495-subclass.json
P1412-subclass.json  P264-subclass.json  P740-subclass.json
P159-subclass.json   P27-subclass.json


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Load tokenizer and model with float16 precision
print("Loading model...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Create a pipeline for text generation
print("Creating pipeline...")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="cuda")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Perform inference with the query
print("Performing inference...")
query = "Which city is Chandos Records's corporate headquarters located?"
# query = "What is the capital city of Hungary?"
output = generator(query, max_length=200, num_return_sequences=1)
print(output)

Loading model...
Free GPU Memory (GB): 23.7812


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.56it/s]


Free GPU Memory (GB): 8.70312
Creating pipeline...
Free GPU Memory (GB): 8.70312


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Performing inference...
[{'generated_text': "Which city is Chandos Records's corporate headquarters located?"}]


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import numpy as np

print("Loading tokenizer and model...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
# model_name = "bigscience/bloomz-560m"
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# model_name = "TinyLlama/TinyLlama_v1.1"
# model_name = "bigscience/bloomz-560m"
model_name = "bigscience/bloomz-1b1"
# model_name = "meta-llama/Meta-Llama-3-8B"
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")

# query = "Which city is Eiffel Tower located in?"
# query = "Spanish. What is the native language of Louis Jules Trochu?"
query = "Which industry does Alan Turing work in?"
# query = "What is the location of Simcoe Composite School?"
input_ids = tokenizer.encode(query, return_tensors='pt').to("cuda")

max_length = 50
generation_config = {
    "temperature": 1,
    "top_p": 0.75,
    "top_k": 40,
    "num_beams": 5,
    "num_return_sequences": 1,
    "output_scores": True,
    "output_hidden_states": False,
    "output_attentions": False,
    "return_dict_in_generate": True
}

print("Generating output...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
with torch.no_grad():
    output_ids = model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=15)

print("Decoding output...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
output_text = tokenizer.decode(output_ids[0][0], skip_special_tokens=True)
print(output_text)

Loading tokenizer and model...
Free GPU Memory (GB): 5.52344
Generating output...
Free GPU Memory (GB): 7.57617
Decoding output...
Free GPU Memory (GB): 7.55078
Which industry does Alan Turing work in? computer scienceI. INTRODUCTION
In recent years, the


In [12]:
from transformers import AutoTokenizer
import transformers 
import torch
model = "TinyLlama/TinyLlama_v1.1"
# model = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model)
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

sequences = pipeline(
    query,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=15,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


Result: Which industry does Alan Turing work in?
Turing worked for Bletchley Park, the headquarters of Britain


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.model.eval()

    def generate_response(self, query, max_new_tokens, temperature):
        input_ids = self.tokenizer.encode(query, return_tensors='pt').to("cuda")
        generation_config = {
            "temperature": temperature,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 1,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }

        with torch.no_grad():
            outputs = self.model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        probabilities = self.extract_probabilities(outputs)
        return output_text, probabilities

    def extract_probabilities(self, outputs):
        probabilities = []
        for score in outputs.scores:
            probs = torch.softmax(score[0], dim=-1)
            top_prob, top_idx = torch.max(probs, dim=-1)
            probabilities.append((self.tokenizer.decode(top_idx), top_prob.item()))
        return probabilities

# Example usage:
model_names = [
    "bigscience/bloomz-560m",
    "bigscience/bloomz-1b1",
    "TinyLlama/TinyLlama_v1.1",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    # "facebook/galactica-1.3b",
    "meta-llama/Meta-Llama-3-8B",
    "meta-llama/Meta-Llama-3-8B-Instruct",
]
queries = [
    "What is the location of Simcoe Composite School?",
    "musical. What is Alan Turing's area of expertise?",
    "Latin. What is the native language of Louis Jules Trochu?"
]
true_answers = [
    "Canada.",
    "logic",
    "French"
]
max_new_tokens_list = [10, 25]
temperature_list = [0.1, 1]

# Initialize a list to store the results
results = []

for model_name in model_names:
    generator = ResponseGenerator(model_name)
    for query, true_answer in zip(queries, true_answers):
        for max_new_tokens in max_new_tokens_list:
            for temperature in temperature_list:
                output_text, probabilities = generator.generate_response(query, max_new_tokens, temperature)
                if output_text.startswith(query):
                    output_text = output_text[len(query):].strip()
                results.append({
                    "Model": model_name,
                    "Query": query,
                    "Max New Tokens": max_new_tokens,
                    "True Answer": true_answer,
                    "Output": output_text,
                    "Probabilities": probabilities
                })

# Convert the results list to a pandas DataFrame
df = pd.DataFrame(results)

# Print the DataFrame
print(df)

# Export the DataFrame to an Excel file
df.to_excel("results.xlsx", index=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.22it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:27<00:00,  6.96s/it]


                                  Model  \
0                bigscience/bloomz-560m   
1                bigscience/bloomz-560m   
2                bigscience/bloomz-560m   
3                bigscience/bloomz-560m   
4                bigscience/bloomz-560m   
5                bigscience/bloomz-560m   
6                 bigscience/bloomz-1b1   
7                 bigscience/bloomz-1b1   
8                 bigscience/bloomz-1b1   
9                 bigscience/bloomz-1b1   
10                bigscience/bloomz-1b1   
11                bigscience/bloomz-1b1   
12             TinyLlama/TinyLlama_v1.1   
13             TinyLlama/TinyLlama_v1.1   
14             TinyLlama/TinyLlama_v1.1   
15             TinyLlama/TinyLlama_v1.1   
16             TinyLlama/TinyLlama_v1.1   
17             TinyLlama/TinyLlama_v1.1   
18   TinyLlama/TinyLlama-1.1B-Chat-v1.0   
19   TinyLlama/TinyLlama-1.1B-Chat-v1.0   
20   TinyLlama/TinyLlama-1.1B-Chat-v1.0   
21   TinyLlama/TinyLlama-1.1B-Chat-v1.0   
22   TinyLl

In [3]:
# Export the DataFrame to an Excel file
df.to_excel("results.xlsx", index=False)

In [5]:
!pwd

/nfs/homedirs/daro/git/quantization-reliability


In [6]:
import shutil
import os

# Define the source and destination paths
source_path = "/nfs/homedirs/daro/git/quantization-reliability/results.xlsx"
destination_path = "results.xlsx"

# Ensure the destination directory exists
# os.makedirs(os.path.dirname(destination_path), exist_ok=True)

# Move the file
shutil.move(source_path, destination_path)

print(f"File moved to {destination_path}")


File moved to results.xlsx
